
# 01 · Dataset & dữ liệu

**Vai trò notebook**: explore VN30 dataset + news coverage + train/test split + lookahead-safe news alignment.

**Owner**: Person 1
**Deadline**: 2026-05-25
**Slide chapter**: 4 — Implementation (Data section)

## Mục tiêu
1. Cho thầy thấy: 5 năm dữ liệu VN30, 5 ticker (VCB / FPT / HPG / VIC / VNM), 248 phiên test.
2. Trình bày news coverage 12 tháng test — bao nhiêu ngày có tin, distribution theo ticker.
3. Chứng minh trực quan: news ngày D chỉ visible từ phiên D+1 (no lookahead).
4. Sample technical indicators (RSI, MACD, Bollinger) trên 1 ticker để minh hoạ feature space.

## Defense Q&A (sẽ trả lời ở cell cuối)
- Q: Tại sao chỉ 5 tickers thay vì cả VN30?
- Q: Train 5 năm có đủ không? Sao không 10 năm?
- Q: News coverage thấp ở ticker nào? Có ảnh hưởng kết quả không?
- Q: Bằng chứng nào để chứng minh không có lookahead trong news?

## Frozen-snapshot rule
Tất cả số liệu trong notebook đọc từ `data/processed/` + `results/`. **Không** chạy lại scraper hay backtest.


## Setup


In [ ]:
import sys
from pathlib import Path

# Make `from _shared import ...` work whether you run from notebooks/ or repo root.
_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from _shared import (  # noqa: E402
    AGENT_COLORS,
    BASELINES,
    DATA,
    FIGURES,
    LLM_AGENTS,
    RESULTS,
    RL_AGENTS,
    ROLE_COLORS,
    TRANSCRIPTS,
    assert_frozen_snapshot,
    list_transcript_dates,
    load_curve,
    load_holdings,
    load_metrics_json,
    load_metrics_table,
    load_transcript,
    save_fig,
    setup_matplotlib,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

setup_matplotlib()
assert_frozen_snapshot()
metrics = load_metrics_table()
print("snapshot OK · agents:", list(metrics.index))



## TODO-01: load-prices — load OHLCV cho 5 tickers
- **OWNER**: Person 1   **DEPENDS**: none
- **READ**: `data/processed/prices.parquet`
- **WRITE**: variable `prices: pd.DataFrame` (index=date, columns=[ticker, open, high, low, close, volume])
- **CONSTRAINTS**:
  - Parse date column
  - Filter chỉ TICKERS = ["VCB", "FPT", "HPG", "VIC", "VNM"]
  - Date range: 2019-01-01 → 2026-04-30 (toàn bộ dataset đã fetch)
- **VALIDATE**:
  - `assert set(prices["ticker"].unique()) == {"VCB", "FPT", "HPG", "VIC", "VNM"}`
  - `assert prices["date"].min() <= pd.Timestamp("2019-01-31")`
  - `assert prices["date"].max() >= pd.Timestamp("2026-04-15")`
  - Print: `prices.groupby("ticker").size()` → mỗi ticker ≥ 1700 phiên
- **PATTERN**: `src/data_pipeline/vnstock_prices.py` cho schema chuẩn
- **DEFENSE Q&A**: "5 tickers chiếm bao nhiêu % vốn hóa VN30?" → cell ngay dưới in marketcap %


In [ ]:
# TODO-01: load 5-ticker prices. Replace `...` with implementation.
prices = ...
# validate
assert set(prices['ticker'].unique()) == {'VCB', 'FPT', 'HPG', 'VIC', 'VNM'}
print(prices.groupby('ticker').size())



## TODO-02: split-windows — đánh dấu train/val/test trên timeline
- **OWNER**: Person 1   **DEPENDS**: TODO-01
- **READ**: `prices` (TODO-01), `src.config` cho TRAIN_START/VAL_START/TEST_START/TEST_END
- **WRITE**: `report/figures/01__split_timeline.png`
- **CONSTRAINTS**:
  - Plot 5 ticker close prices stacked or in subplots
  - Shade 3 region: train (xanh nhạt), val (vàng nhạt), test (cyan đậm)
  - Annotate boundary dates trên trục x
  - Title VI: "Phân chia train / validation / test trên 5 năm dữ liệu VN30"
- **VALIDATE**:
  - `assert (FIGURES / "01__split_timeline.png").exists()`
  - Visual sanity: 3 vùng phân biệt rõ
- **PATTERN**: pattern matplotlib `axvspan(..., alpha=0.15)` cho region highlight
- **DEFENSE Q&A**: "Tại sao chia train 5 năm + val 4 tháng + test 12 tháng?" → Markdown cell dưới chart trả lời


In [ ]:
# TODO-02: split timeline figure
# Hint: use plt.axvspan for each window, plt.savefig via save_fig('01__split_timeline')
pass



## TODO-03: news-coverage — heatmap mật độ tin theo ticker × tháng
- **OWNER**: Person 1   **DEPENDS**: none
- **READ**: `data/processed/news.parquet`
- **WRITE**: `report/figures/01__news_coverage_heatmap.png` + table `news_coverage_pct`
- **CONSTRAINTS**:
  - Group news theo (year-month, ticker) → đếm số tin
  - Heatmap: rows=ticker, cols=12 tháng test (2025-05 → 2026-04)
  - Color: cyan gradient (light = ít tin, đậm = nhiều)
  - Annotate cell values (số tin)
  - Title VI: "Mật độ tin tức 12 tháng test trên 5 ticker VN30"
- **VALIDATE**:
  - `coverage_pct = (df > 0).sum().sum() / df.size`
  - `print(f"Coverage: {coverage_pct:.1%}")` → kỳ vọng ≥ 50% (từ checkpoint 16/05)
  - File exists
- **PATTERN**: `scripts/news_coverage_report.py` đã có code aggregate sẵn
- **DEFENSE Q&A**: "Coverage 50% có đủ tin cậy cho LLM agent không?" → ref bảng coverage_pct


In [ ]:
# TODO-03: news coverage heatmap
news = pd.read_parquet(DATA / 'news.parquet')
# ...



## TODO-04: news-shift-proof — chứng minh news ngày D chỉ visible D+1 close
- **OWNER**: Person 1   **DEPENDS**: TODO-03
- **READ**: `news` (TODO-03), 1 ticker mẫu (VCB), 5 ngày liên tiếp
- **WRITE**: bảng markdown 5 rows × 4 cols [date_published, ticker, headline, visible_from]
- **CONSTRAINTS**:
  - `visible_from = published_date + 1 trading day` (skip weekend/holiday)
  - Highlight ô khi `visible_from > published_date` (luôn là true)
  - 1 markdown cell mở đầu giải thích: "Mỗi news được shift D→D+1 close để tránh lookahead bias"
  - 1 code cell render bảng
- **VALIDATE**:
  - `assert (df['visible_from'] > df['published_date']).all()`
- **PATTERN**: `src/data_pipeline/news_align.py:shift_news_to_dplus1` — function đó CHÍNH XÁC làm việc này
- **DEFENSE Q&A** (critical): "Cơ chế nào đảm bảo backtest không leak future news?" → ref function + table này


In [ ]:
# TODO-04: news shift proof — show 5 sample rows with visible_from = published + 1
pass



## TODO-05: indicators-sample — vẽ RSI / MACD / Bollinger cho VCB
- **OWNER**: Person 1   **DEPENDS**: TODO-01
- **READ**: `prices` filtered to VCB, full date range
- **WRITE**: `report/figures/01__indicators_vcb.png` (3-row subplot: price + Bollinger, RSI, MACD)
- **CONSTRAINTS**:
  - Top: VCB close + Bollinger upper/lower (20-period)
  - Middle: RSI(14) với guidelines 30/70
  - Bottom: MACD(12,26,9) + signal + histogram
  - Time range: chỉ vẽ test window 2025-05 → 2026-04 cho rõ
  - Title VI: "Chỉ báo kỹ thuật mẫu — VCB trong giai đoạn test"
- **VALIDATE**:
  - File exists, ≥ 100KB
  - RSI values ∈ [0, 100]
  - MACD signal sign-changes matching expected market regime
- **PATTERN**: `ta` library — `ta.momentum.RSIIndicator`, `ta.trend.MACD`, `ta.volatility.BollingerBands`
- **DEFENSE Q&A**: "Agents thấy được những features gì khi quyết định?" → ref figure này


In [ ]:
# TODO-05: indicators chart for VCB
import ta
# ...



## TODO-06: data-summary-table — bảng tóm tắt cho slide
- **OWNER**: Person 1   **DEPENDS**: TODO-01, TODO-03
- **WRITE**: `report/figures/01__data_summary.md` (markdown table file, Person 1 paste vào slide)
- **CONSTRAINTS**:
  Bảng 6 rows × 2 cols:
  | Metric | Giá trị |
  | ----- | ----- |
  | Tickers | VCB, FPT, HPG, VIC, VNM |
  | Train | 2019-01 → 2024-12 (X phiên) |
  | Val | 2025-01 → 2025-04 (X phiên) |
  | Test | 2025-05 → 2026-04 (248 phiên) |
  | News rows | X tin sau D+1 shift |
  | News coverage | XX% test days have ≥1 news |
- Read số liệu từ DataFrame vừa load, không hardcode
- **VALIDATE**: file đọc lại được, đủ 6 rows
- **DEFENSE Q&A**: "Tổng quan dataset như nào?" → mở figure này hoặc đọc table


In [ ]:
# TODO-06: write data summary markdown table
summary_path = FIGURES / '01__data_summary.md'
# ...



## Defense Q&A — câu trả lời sẵn

> **Q1: Tại sao chỉ 5 tickers thay vì cả VN30?**
> A: Trade-off chất lượng vs số mã. 5 mã top market-cap đảm bảo (a) liquidity đủ cho ±7% lot-100 execution không bị slippage, (b) news coverage cao (large-cap nhiều tin), (c) state space đủ small để DDPG converge trong 5 năm train.
> Evidence: TODO-01 marketcap %, TODO-03 coverage heatmap.

> **Q2: Train 5 năm có đủ không?**
> A: 1750+ daily samples + 5 tickers = ~8750 observations. PPO converge sau ~200K steps theo training log. DDPG saturate tanh (Risk #7 PRD §14), backup bằng PPO.
> Evidence: `results/{ddpg,ppo}_training_log.jsonl`.

> **Q3: News coverage thấp ảnh hưởng kết quả?**
> A: Coverage ~XX% (TODO-03). LLM agent fallback `hold` khi không có news → parse_failure_rate=0 trên multi_agent.
> Evidence: `results/multi_agent/metrics.json` field `parse_failure_rate`.

> **Q4: Chứng minh không lookahead news?**
> A: Mỗi row news có `visible_from = published_date + 1 trading day`. `_get_state()` của VNTradingEnv filter `news[news.visible_from <= current_date]` trước khi expose cho agent.
> Evidence: TODO-04 + `src/data_pipeline/news_align.py:shift_news_to_dplus1` + `tests/test_news_align.py`.
